In [41]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config
from tempfile import TemporaryDirectory
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
import xgboost as xgb
from xgboost import XGBClassifier
#from lightgbm import LGBMClassifier
#from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn import metrics
import seaborn as sns
import numpy as np
import warnings

In [42]:
target_column = "health_condition"
df = pd.read_csv("data/train.csv")

In [43]:
df.drop('id', axis=1, inplace=True)

df['diet_type'] = df['diet_type'].astype('category')
df['gender'] = df['gender'].astype('category')
df['stress_level'] = df['stress_level'].replace({'low':0, 'medium':1, 'high':2})
df['sleep_quality'] = df['sleep_quality'].replace({'poor':0, 'average':1, 'good':2})
df['physical_activity_level'] = df['physical_activity_level'].replace({'sedentary':0, 'moderate':1, 'active':2})
df['smoking_alcohol'] = df['smoking_alcohol'].replace({'no':0, 'occasional':1, 'yes':2})

object_cols = ['stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol']
for col in object_cols:
    df[col] = df[col].astype(np.float64)

In [44]:
df['calorie_expenditure_per_step'] = df['calorie_expenditure'] / (df['step_count']+1)
df['step_speed'] = df['step_count'] / (df['exercise_duration']+1)
df['calorie_expenditure_per_min'] = df['calorie_expenditure'] / (df['exercise_duration']+1)
df['calorie_expenditure_per_bmi'] = df['calorie_expenditure'] / (df['bmi']+1)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 18 columns):
 #   Column                        Non-Null Count   Dtype   
---  ------                        --------------   -----   
 0   health_condition              690088 non-null  str     
 1   sleep_duration                614089 non-null  float64 
 2   heart_rate                    682255 non-null  float64 
 3   bmi                           676190 non-null  float64 
 4   calorie_expenditure           637235 non-null  float64 
 5   step_count                    676172 non-null  float64 
 6   exercise_duration             683187 non-null  float64 
 7   water_intake                  646611 non-null  float64 
 8   diet_type                     683187 non-null  category
 9   stress_level                  607277 non-null  float64 
 10  sleep_quality                 631757 non-null  float64 
 11  physical_activity_level       653467 non-null  float64 
 12  smoking_alcohol               661506 non-

In [45]:
y = df[target_column].replace({'unhealthy':0, 'at-risk':1, 'fit':2}).astype('int')
X = df.drop(target_column, axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)